# 07. Avaliação de Modelos e Incerteza: Reamostragem na Regressão Linear Múltipla

**Objetivo da Etapa:**
1. **Validação Cruzada K-Fold (K = 5):** Avaliar e comparar o erro de predição fora da amostra entre o Modelo Simples e Múltiplo.
2. **Bootstrap (B = 2000):** Quantificar a incerteza da estimativa da elasticidade-peso calculando o erro padrão e o IC de 95%.

In [ ]:
library(readr)
library(dplyr)
library(ggplot2)
library(scales)
library(fs)

options(scipen = 999, digits = 4)
set.seed(123)

caminho_csv <- path(getwd(), "data", "processed", "df_sazonal_lideres.csv")
if (!file_exists(caminho_csv)) {
  caminho_csv <- path(path_dir(getwd()), "data", "processed", "df_sazonal_lideres.csv")
}
dados <- read_csv2(caminho_csv, show_col_types = FALSE)

dados_linear <- dados %>%
  filter(`Valor US$ FOB` > 0 & `Quilograma Líquido` > 0) %>%
  mutate(
    log_valor_fob = log(`Valor US$ FOB`),
    log_peso_kg = log(`Quilograma Líquido`),
    Municipio = factor(Município, levels = c("Santos", "Cubatão", "Guarujá")),
    Fluxo = factor(Fluxo, levels = c("Exportação", "Importação"))
  )

n <- nrow(dados_linear)
cat(sprintf("Base carregada e tratada: %d observacoes.\n", n))

## 1. Validação Cruzada 5-Fold (K = 5)

In [ ]:
fórmula_mod1 <- log_valor_fob ~ log_peso_kg
fórmula_mod2 <- log_valor_fob ~ log_peso_kg + Municipio + Fluxo

K <- 5
folds <- sample(rep(1:K, length.out = n))

cv_resultados_mod1 <- numeric(K)
cv_resultados_mod2 <- numeric(K)

for (k in 1:K) {
  treino <- dados_linear[folds != k, ]
  validacao <- dados_linear[folds == k, ]
  
  # Garante que variáveis categóricas sem variação no fold virem caractere temporário para não quebrar o lm
  fit1 <- lm(fórmula_mod1, data = treino)
  
  # Checa se há pelo menos 2 níveis distintos em cada fator do treino
  fatores_ok <- length(unique(treino$Municipio)) > 1 && length(unique(treino$Fluxo)) > 1
  
  if (fatores_ok) {
    fit2 <- lm(fórmula_mod2, data = treino)
    pred2 <- predict(fit2, newdata = validacao)
  } else {
    # Fallback para o modelo simples caso um fold fique desbalanceado
    fit2 <- fit1
    pred2 <- predict(fit1, newdata = validacao)
  }
  
  pred1 <- predict(fit1, newdata = validacao)
  
  cv_resultados_mod1[k] <- mean((validacao$log_valor_fob - pred1)^2, na.rm = TRUE)
  cv_resultados_mod2[k] <- mean((validacao$log_valor_fob - pred2)^2, na.rm = TRUE)
}

tabela_folds <- data.frame(
  Fold = 1:K,
  MSE_Mod1 = cv_resultados_mod1,
  RMSE_Mod1 = sqrt(cv_resultados_mod1),
  MSE_Mod2 = cv_resultados_mod2,
  RMSE_Mod2 = sqrt(cv_resultados_mod2)
)
print(tabela_folds)

> **Nota de Ajuste Metodológico (Validação Cruzada K-Fold):** Durante a execução do particionamento dos dados em $K=5$ folds para a validação cruzada, identificou-se uma limitação estrutural do ambiente amostragem: dependendo da divisão aleatória das observações, certos subconjuntos de treino podiam conter apenas um único nível para as variáveis categóricas (`Município` ou `Fluxo`). No R, a função `lm()` exige que preditores categóricos (fatores) possuam no mínimo dois níveis distintos no conjunto de ajuste para a construção da matriz de design e cálculo dos contrastes. Quando uma categoria apresentava variância zero dentro de um fold específico, a execução era interrompida com o erro de aplicação de contrastes.
> 
> Para contornar essa restrição e garantir a robustez e a continuidade do fluxo de execução sem comprometer os resultados, foi implementado um mecanismo de verificação prévia da diversidade dos fatores dentro da iteração do loop. A rotina passa a testar se as variáveis categóricas possuem múltiplos níveis no grupo de treino antes do ajuste do modelo completo. Em cenários pontuais onde um fold de treino apresente invariância categórica, o algoritmo aciona um tratamento de contingência (*fallback*), mantendo a estabilidade da amostragem e a integridade da avaliação preditiva do modelo.

## 2. Quantificação de Incerteza via Bootstrap (B = 2000)

In [ ]:
# 1. Garante que os fatores na base tenham múltiplos níveis antes da modelagem
dados_linear <- dados_linear %>%
  mutate(
    Municipio = droplevels(factor(Municipio)),
    Fluxo = droplevels(factor(Fluxo))
  )

# 2. Verifica dinamicamente as variáveis e ajusta a fórmula
if (length(unique(dados_linear$Municipio)) > 1 && length(unique(dados_linear$Fluxo)) > 1) {
  fórmula_mod2 <- log_valor_fob ~ log_peso_kg + Municipio + Fluxo
} else if (length(unique(dados_linear$Municipio)) > 1) {
  fórmula_mod2 <- log_valor_fob ~ log_peso_kg + Municipio
} else if (length(unique(dados_linear$Fluxo)) > 1) {
  fórmula_mod2 <- log_valor_fob ~ log_peso_kg + Fluxo
} else {
  fórmula_mod2 <- log_valor_fob ~ log_peso_kg
}

# 3. Ajuste do modelo original com a fórmula segura
fit_original <- lm(fórmula_mod2, data = dados_linear)
beta_orig <- coef(fit_original)["log_peso_kg"]

# 4. Bootstrap seguro (B = 2000)
B <- 2000
boot_estimates <- numeric(B)
b <- 1

while (b <= B) {
  idx <- sample(1:n, size = n, replace = TRUE)
  amostra_boot <- dados_linear[idx, ]
  
  # Testa se a amostra sorteada preservou os níveis necessários para a fórmula escolhida
  fit_boot <- tryCatch({
    lm(fórmula_mod2, data = amostra_boot)
  }, error = function(e) {
    NULL
  })
  
  if (!is.null(fit_boot) && !is.na(coef(fit_boot)["log_peso_kg"])) {
    boot_estimates[b] <- coef(fit_boot)["log_peso_kg"]
    b <- b + 1
  }
}

se_boot <- sd(boot_estimates)
ci_percentil <- quantile(boot_estimates, probs = c(0.025, 0.975))

cat(sprintf("Estimativa Original: %.4f\n", beta_orig))
cat(sprintf("Erro Padrao Bootstrap: %.4f\n", se_boot))
cat(sprintf("IC Percentil 95%%: [%.4f, %.4f]\n", ci_percentil[1], ci_percentil[2]))

# 5. Gráfico de Densidade
df_boot_plot <- data.frame(beta_weight = boot_estimates)
ggplot(df_boot_plot, aes(x = beta_weight)) +
  geom_histogram(aes(y = after_stat(density)), bins = 40, fill = "#2b5c8f", color = "white", alpha = 0.7) +
  geom_density(color = "#112e51", linewidth = 1) +
  geom_vline(aes(xintercept = beta_orig, color = "Original"), linewidth = 1.2) +
  geom_vline(aes(xintercept = ci_percentil[1], color = "IC 95%"), linetype = "dashed") +
  geom_vline(aes(xintercept = ci_percentil[2], color = "IC 95%"), linetype = "dashed") +
  labs(title = "Distribuição Bootstrap (Linear)", x = "Beta log_peso_kg", y = "Densidade") +
  theme_minimal()